Open a Open-X Embodiment (OxE) dataset

* Goal:
    * Open OxE dataset.
    * Inspect its content.
    * Plot: images, time series of actions, etc.
    * Save a subpart of it (5%) in hdf5 format.
    * Open hdf5 saved file -> Explore whether the format is the same as TFDS format.
    * Save two datasets together.
* notes:
    - Data heterogeneity:
        - Images have different resolutions.
        - Action can represent absolute positions/orientations or delta positions/orientations.
* environment: oxe_download (env_oxe_download.yml)
* OxE datasets: https://robotics-transformer-x.github.io/

In [9]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import h5py
import numpy as np
import json

import tensorflow_datasets as tfds
import tensorflow as tf

## Load a dataset

<b>Notes</b>:

* dataset = builder.as_dataset(split="train[:N]") -> selects N episodes from the builder.
* dataset.take(Nn) -> Take Nn episodes from the outer pool of N episodes.

<b>builder_dir</b>:
 
* All listed datasets in: https://docs.google.com/spreadsheets/d/1rPBD77tk60AEIGZrGSODwyyzs5FgCU9Uz3h-3_t2A9g/edit?gid=0#gid=0
* gs://gresearch/robotics/<dataset_name>/<version>
    * dataset_name: name in the column S (of the excel sheet) labeled "_Registered Dataset Name_"
    * version:
        * Most datasets: 0.1.0
        * robo_net: 1.0.0
        * language_table: 0.0.1
        * droid: 1.0.0 or 1.0.1
* Examples used:
    * "gs://gresearch/robotics/bridge/0.1.0"

* Action space: Datasets can represent absolute value, the delta change, or the velocity of the dimension

In [2]:
def feature_to_metadata(feature):
    # Extract feature metadata from the builder.
    # Use to get the metadata about the actions and transform it into dictionary for saving with hdf5.
    if isinstance(feature, tfds.features.FeaturesDict):
        return {
            key: feature_to_metadata(value)
            for key, value in feature.items()
        }

    metadata = {
        "type": type(feature).__name__,
    }

    if hasattr(feature, "shape") and feature.shape is not None:
        metadata["shape"] = list(feature.shape)

    if hasattr(feature, "dtype") and feature.dtype is not None:
        metadata["dtype"] = str(feature.dtype)

    if hasattr(feature, "doc") and feature.doc:
        metadata["description"] = str(feature.doc)

    if isinstance(feature, tfds.features.ClassLabel):
        metadata["num_classes"] = feature.num_classes
        metadata["names"] = feature.names

    return metadata

In [ ]:
# builder_dir = "gs://gresearch/robotics/ucsd_kitchen_dataset_converted_externally_to_rlds/0.1.0"
# builder_dir = "gs://gresearch/robotics/robo_net/0.1.0"

def open_dataset(builder_dir: str, n_episodes: int, verbose=True):
    builder = tfds.builder_from_directory(
        builder_dir=builder_dir
    )

    print("====================================================")

    action_feature = builder.info.features["steps"]["action"]

    if verbose:
        print(f"Action feature:\n{action_feature}")

    # Convert TFDS feature objects into plain Python dictionaries
    action_metadata = feature_to_metadata(action_feature)

    if verbose:
        print("Action metadata:")
        print(action_metadata)

    dataset = builder.as_dataset(
        split=f"train[:{n_episodes}]"
    )

    print(
        f"Data n_episodes on disk: "
        f"{builder.info.dataset_size / 1e9:.2f} GB"
    )

    return dataset, action_metadata

data_name_lst = [
    "stanford_hydra_dataset_converted_externally_to_rlds",
    "taco_play"
] # datasets characteristics: Franka, Single Arm, EEF Position.

data_lst = [open_dataset(builder_dir=f"gs://gresearch/robotics/{data_name}/0.1.0",n_episodes=1)[0] for data_name in data_name_lst]
metadata_lst = [open_dataset(builder_dir=f"gs://gresearch/robotics/{data_name}/0.1.0",n_episodes=1)[1] for data_name in data_name_lst]

Action feature:
Tensor(shape=(7,), dtype=float32, description=Robot action, consists of [3x EEF positional delta, 3x EEF orientation delta in euler angle, 1x close gripper].)
Action metadata:
{'type': 'Tensor', 'shape': [7], 'dtype': "<dtype: 'float32'>", 'description': "Documentation(desc='Robot action, consists of [3x EEF positional delta, 3x EEF orientation delta in euler angle, 1x close gripper].', value_range='')"}
Data size on disk: 77.83 GB


Action feature:
FeaturesDict({
    'actions': Tensor(shape=(7,), dtype=float32, description=absolute desired values for gripper pose (first 6 dimensions are x, y, z, yaw, pitch, roll), last dimension is open_gripper (-1 is open gripper, 1 is close)),
    'rel_actions_gripper': Tensor(shape=(7,), dtype=float32, description=relative actions for gripper pose in the gripper camera frame (first 6 dimensions are x, y, z, yaw, pitch, roll), last dimension is open_gripper (-1 is open gripper, 1 is close)),
    'rel_actions_world': Tensor(shape=(7,), dtype=float32, description=relative actions for gripper pose in the robot base frame (first 6 dimensions are x, y, z, yaw, pitch, roll), last dimension is open_gripper (-1 is open gripper, 1 is close)),
    'terminate_episode': float32,
})
Action metadata:
{'actions': {'type': 'Tensor', 'shape': [7], 'dtype': "<dtype: 'float32'>", 'description': "Documentation(desc='absolute desired values for gripper pose (first 6 dimensions are x, y, z, yaw, pitc

Action feature:
Tensor(shape=(7,), dtype=float32, description=Robot action, consists of [3x EEF positional delta, 3x EEF orientation delta in euler angle, 1x close gripper].)
Action metadata:
{'type': 'Tensor', 'shape': [7], 'dtype': "<dtype: 'float32'>", 'description': "Documentation(desc='Robot action, consists of [3x EEF positional delta, 3x EEF orientation delta in euler angle, 1x close gripper].', value_range='')"}
Data size on disk: 77.83 GB


Action feature:
FeaturesDict({
    'actions': Tensor(shape=(7,), dtype=float32, description=absolute desired values for gripper pose (first 6 dimensions are x, y, z, yaw, pitch, roll), last dimension is open_gripper (-1 is open gripper, 1 is close)),
    'rel_actions_gripper': Tensor(shape=(7,), dtype=float32, description=relative actions for gripper pose in the gripper camera frame (first 6 dimensions are x, y, z, yaw, pitch, roll), last dimension is open_gripper (-1 is open gripper, 1 is close)),
    'rel_actions_world': Tensor(shape=(7,), dtype=float32, description=relative actions for gripper pose in the robot base frame (first 6 dimensions are x, y, z, yaw, pitch, roll), last dimension is open_gripper (-1 is open gripper, 1 is close)),
    'terminate_episode': float32,
})
Action metadata:
{'actions': {'type': 'Tensor', 'shape': [7], 'dtype': "<dtype: 'float32'>", 'description': "Documentation(desc='absolute desired values for gripper pose (first 6 dimensions are x, y, z, yaw, pitc

## Inspect data

<b>Notes</b>:

##### Bridge dataset
| Key | Value |
|---|---|
| Episode | `dict_keys(['steps'])` |
| Step | `dict_keys(['action', 'is_first', 'is_last', 'is_terminal', 'observation', 'reward'])` |
| Observation | `dict_keys(['image', 'natural_language_embedding', 'natural_language_instruction', 'state'])` |
| Action | `dict_keys(['open_gripper', 'rotation_delta', 'terminate_episode', 'world_vector'])` |

---

| Key | Meaning |
|---|---|
| world_vector | `The translation part of the action: a 3D vector [dx, dy, dz] representing how much to move the end-effector, expressed in the world/base frame (not the camera or gripper's local frame)` |
| rotation_delta | `The rotation part of the action: how much to rotate the end-effector, usually as a 3-element delta (roll, pitch, yaw) or axis-angle representation, also relative to world/base frame` |
| open_gripper | `Boolean/scalar gripper command — whether to open (vs. close) the gripper at this timestep` |
| terminate_episode | `Flag/scalar indicating whether the policy is signaling "I'm done with the task now" — distinct from is_last/is_terminal at the step level, which mark the actual end of the recorded trajectory` |

* Structure: Episode -> Steps -> Observations/Actions.
* More on dataset format: https://github.com/google-research/rlds#dataset-format

In [4]:
# Pull one episode and look at its structure

def _print_keys_values_dataset(data, data_name: str):
    print(f"\n\nDataset: {data_name}")
    for episode in data.take(1):
        print(f"Episode keys: \t\t{episode.keys()}")
        print(f"Number of steps: \t{len(episode['steps'])}")
        for step in episode["steps"].take(1):
            print(f"Step keys: \t\t{step.keys()}")
            print(f"is_first: \t\t{step['is_first']}")
            print(f"is_last: \t\t{step['is_last']}")
            print(f"is_terminal: \t\t{step['is_terminal']}")
            print(f"reward: \t\t{step['reward']}")
            
            
            print("\n================== Observation ========================")
            print(f"Observation keys: \t{step['observation'].keys()}")
            for key_obs in list(step['observation'].keys()):
                print(f"================== {key_obs}")
                print(step['observation'][key_obs])
                print("\n")

            print("========================================================================")
            print("\n\n================== Action ========================")
            if isinstance(step['action'],dict):
                print(f"Action keys: \t\t{step['action'].keys()}")
                for key_act in list(step['action'].keys()):
                    print(f"================== {key_act}")
                    print(step['action'][key_act])
                    print("\n")
            else:
                print(f"Action: \t\t{step['action'].numpy()}")

            print("\n=========================== Step end ===================================")
            print("========================================================================")
        
        print("=========================== Episode end ===================================")
        print("========================================================================\n")

In [15]:
for episode in data_lst[1].take(1):
    print(f"Episode keys: \t\t{episode.keys()}")
    print(f"Number of steps: \t{len(episode['steps'])}")
    for step in episode["steps"].take(1):
        print(f"Step keys: \t\t{step.keys()}")


Episode keys: 		dict_keys(['steps'])
Number of steps: 	66
Step keys: 		dict_keys(['action', 'is_first', 'is_last', 'is_terminal', 'observation', 'reward'])


In [16]:
any("language" in key for key in step.keys())

False

In [10]:
for idx, data in enumerate(data_lst): _print_keys_values_dataset(data, data_name=data_name_lst[idx])



Dataset: stanford_hydra_dataset_converted_externally_to_rlds
Episode keys: 		dict_keys(['episode_metadata', 'steps'])
Number of steps: 	549
Step keys: 		dict_keys(['action', 'discount', 'is_dense', 'is_first', 'is_last', 'is_terminal', 'language_embedding', 'language_instruction', 'observation', 'reward'])
is_first: 		True
is_last: 		False
is_terminal: 		False
reward: 		0.0

================== Observation ========================
Observation keys: 	dict_keys(['image', 'state', 'wrist_image'])
================== image
tf.Tensor(
[[[ 36  36  29]
  [ 35  35  28]
  [ 33  33  31]
  ...
  [ 86  82  80]
  [ 83  82  80]
  [ 86  84  82]]

 [[ 36  35  31]
  [ 35  34  29]
  [ 31  32  30]
  ...
  [ 86  82  80]
  [ 83  81  81]
  [ 83  81  81]]

 [[ 39  35  33]
  [ 37  33  31]
  [ 33  32  30]
  ...
  [ 82  81  79]
  [ 83  82  80]
  [ 83  82  80]]

 ...

 [[170 203 234]
  [177 210 241]
  [169 202 233]
  ...
  [ 91 132 166]
  [ 97 133 168]
  [ 96 132 167]]

 [[174 207 244]
  [163 196 234]
  [151 191

## Plot dataset

#### Image

* Take episode 1.
* Take all steps in episode 1.
* Plot an image per step.

In [ ]:
def _plot_image_from_dataset(data, data_name: str):
    print(f"\n\nDataset: {data_name}")
    n_steps = len(next(iter(data.take(1)))["steps"])
    print(f"length episode 1: {n_steps}")
    if n_steps > 50:
        n_steps = 12 # Put a maximum of steps per episode, in case there are a lot of them.

    for episode in data.take(1):
        for step in episode["steps"].take(n_steps):
            if "image" in step['observation'].keys():
                img = step["observation"]["image"]
                plt.imshow(img.numpy())
                plt.show()
            else:
                img = step["observation"]["rgb_gripper"] #  rgb_gripper or rgb_static
                plt.imshow(img.numpy())
                plt.show()

In [ ]:
idx = 0
_plot_image_from_dataset(data_lst[idx], data_name = data_name_lst[idx])

#### Action sequence

In [ ]:
for episode in dataset.take(1):
    rotation_delta, world_vector = np.zeros(shape=(n_steps,3)), np.zeros(shape=(n_steps,3))
    idx = 0
    for step in episode["steps"].take(n_steps):
        print(f"idx: {idx} --- \t\t\t --- {step['action']['rotation_delta'].numpy()}")
        rotation_delta[idx,:] = step["action"]["rotation_delta"].numpy()
        world_vector[idx,:] = step["action"]["world_vector"].numpy()
        idx += 1

In [ ]:
'''TODO: 
    - add labels and titles:
        -- x_label: steps.
        -- y_label: units of world vector and rotation_delta --> TODO: look at this.
        -- subtitles: each dimensions corresponding to each component of world vector and rotation delta.
'''

fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(9,5))

for dim in range(3):
    ax[0,dim].scatter(np.arange(n_steps), rotation_delta[:,dim])
    ax[1,dim].scatter(np.arange(n_steps), world_vector[:,dim])

fig.tight_layout()
plt.show()

#### Action histogram

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(9,5))

for dim in range(3):
    ax[0,dim].hist(rotation_delta[:,dim])
    ax[1,dim].hist(world_vector[:,dim])

fig.tight_layout()
plt.show()

## Save a fraction of dataset

<b>Notes</b>:

* dataset.save() vs builder.download_and_prepare():
    * dataset.save() is a much lighter, TF-native way to freeze a specific slice or transformed pipeline to disk quickly, but it loses TFDS-level metadata like dataset_size, feature descriptions, citation, etc.
    * download_and_prepare() gives the whole  dataset back in TFDS's own format (portable, versioned, reloadable with all metadata).
* Save hdf5:
    * Goal: Upload dataset with environments that does not contain Tensorflow (tensorflow can have incompatible libraries with pytorch).

In [17]:
def _get_candidate_keys(dimension: str):
    # Get keys for actions and images depending on datasets

    if dimension == "image":
        return ["image", "rgb_gripper", "rgb_static"]
    elif dimension == "language":
        return ["language_instruction", "natural_language_instruction"]
    # TODO: Incorporate a ValueError loop -> dimension has to be "image" or "language"

In [ ]:
#### save data slice as hdf5

# transform into hdf5 format. Useful for uploading with an environment without tensorflow.
def save_hdf5_dataset(path_data: str, data_name_lst: list, n_episodes: int):
    with h5py.File(f"{path_data}/combined_datasets.h5", "w") as f:
        for data_name in data_name_lst:
            print("\n\n====================================================")
            print(f"Dataset: {data_name}")
            dataset, action_metadata = open_dataset(builder_dir=f"gs://gresearch/robotics/{data_name}/0.1.0", n_episodes=n_episodes, verbose=False)
            dataset_grp = f.create_group(f"{data_name}")

            for i, episode in enumerate(dataset.take(n_episodes)):
                grp_episode = dataset_grp.create_group(f"episode_{i}")
                steps = list(episode["steps"])

                # Collect across all steps first
                ## images
                candidate_image_keys = _get_candidate_keys(dimension="image") # keys that might contain images.
                grp_images = grp_episode.create_group("images")
                for key in candidate_image_keys:
                    if key in steps[0]['observation'].keys():
                        images = np.stack([s["observation"][key].numpy() for s in steps])
                        grp_images.create_dataset(key, data=images)

                ## actions
                grp_action = grp_episode.create_group("actions")
                if isinstance(steps[0]["action"], dict):
                    for key in steps[0]['action'].keys():
                        action = np.stack([s["action"][key].numpy() for s in steps])
                        grp_action.create_dataset(key, data=action)
                else:
                    action = np.stack([s["action"].numpy() for s in steps])
                    grp_action.create_dataset("actions", data=action)
                
                ## language
                candidate_language_keys = _get_candidate_keys(dimension="language")
                grp_language = grp_episode.create_group("language")
                if any("language" in key for key in steps[0].keys()):
                    for key in candidate_language_keys:
                        if key in steps[0].keys():
                            language_instructions = np.stack([s[key].numpy() for s in steps])
                            grp_language.create_dataset(key, data=language_instructions)
                else:
                    for key in candidate_language_keys:
                        if key in steps[0]['observation'].keys():
                            language_instructions = np.stack([s["observation"][key].numpy() for s in steps])
                            grp_language.create_dataset(key, data=language_instructions)
            
            dataset_grp.attrs["actions_metadata"] = json.dumps(action_metadata)
    
    # Print data size
    file_path = Path(f"{path_data}/combined_datasets.h5")
    size_bytes = file_path.stat().st_size
    print(f"Size of file {file_path}: {size_bytes / (1024 ** 2):.2f} MB")
    print(f"Size of file {file_path}: {size_bytes / (1024 ** 3):.2f} GB")

In [ ]:
path_data = str(Path.cwd().parent / "data")
save_hdf5_dataset(path_data=path_data, 
                  data_name_lst=data_name_lst,
                  n_episodes=16)



Dataset: stanford_hydra_dataset_converted_externally_to_rlds
Data size on disk: 77.83 GB


Dataset: taco_play
Data size on disk: 51.29 GB
Size of file /home/fiorenza/DELL/goettingen/1_work/project_repositories/VLA_FrankaArm/data/combined_datasets.h5: 2257.61 MB
Size of file /home/fiorenza/DELL/goettingen/1_work/project_repositories/VLA_FrankaArm/data/combined_datasets.h5: 2.20 GB
